# ml_16 : rattrapage des groupes à population faible (étape 5 de la phase finale)

**But** : reprendre, avec une comparaison plus fine, les groupes de colonnes écartés « dans le bruit » **parce que trop peu de clients sont concernés** pour que la comparaison des scénarios puisse voir leur effet (`methodologie_ML.md`, section 9 : moins de 2 % des clients du train). Un seul groupe est dans ce cas, d'après le journal `tableau_ML.md` : les **traces des corrections** de `ml_2` (`FAUX_CODAGE`, `SURVEILLANCE_RECENTE`), écarté « à reprendre en phase finale ».

**Ce n'est pas un scénario** : rien n'est écrit dans le fichier de résultats, aucun modèle n'est enregistré, le test n'est pas lu. La décision (garder ou non le groupe) s'écrit ensuite à la main dans le journal.

**Protocole, fixé avant de lancer** (la méthodologie laissait la comparaison fine à décider à ce moment-là) :
1. **Base** : la version retenue à l'issue du duel de l'étape 4 (`BASE` : `"14"` ou `"15"`, cellule de réglages), avec ses **réglages figés**, relus dans le fichier de résultats (pas de nouvelle recherche d'hyperparamètres).
2. **Deux versions comparées** : la base **avec** et **sans** le groupe. Si la base ne contient pas le groupe (`ml_14`), c'est un test d'**ajout** ; si elle le contient déjà (`ml_15`, colonnes de `ml_11`), c'est un test de **retrait**. Dans les deux cas, on mesure ce que le groupe apporte.
3. **Validation croisée répétée** : 5 plis × 10 répétitions (`RepeatedStratifiedKFold`, `random_state=42`), soit 50 comparaisons pli par pli, au lieu de 5 dans les scénarios. Les deux versions voient exactement les mêmes plis.
4. **Mesure** : le score décisionnel F2 au rappel minimal de 60 % (celui des scénarios), et la précision moyenne en contrôle.
5. **Verdict** : sur les 50 gains pli par pli (avec le groupe moins sans), un **test t corrigé de Nadeau et Bengio**. Les répétitions réutilisent les mêmes clients : leurs résultats sont corrélés, et un test t ordinaire serait trop optimiste. La correction multiplie la variance par (1 / 50 + 1 / 4), le second terme étant le rapport entre la taille d'un pli de validation et celle de la partie d'entraînement. **Gain réel** si t ≥ 2 ; **perte** si t ≤ -2 ; sinon **dans le bruit**. Pour le meilleur contre meilleur, chaque version prend son meilleur modèle, sur la moyenne des 50 plis.
6. **Décision** : le groupe est gardé s'il fait un **gain réel en meilleur contre meilleur** ; sinon il est écarté définitivement.

**Données** : `data/ML/train_dataset.csv` (clients au contentieux à M retirés, 80 % des clients restants). Le test n'est pas lu.

In [1]:
# ==============================================================================
# RÉGLAGES DU LANCEMENT
# ==============================================================================

# Version de base : celle retenue à l'issue du duel de l'étape 4 ("14" : base ml_12 ; "15" : colonnes de ml_11), à mettre à jour avant de lancer
BASE = "14"

# Groupe à population faible repris (journal : ml_2, « à reprendre en phase finale »)
GROUPE = ["FAUX_CODAGE", "SURVEILLANCE_RECENTE"]

# Comparaison fine : 5 plis × N_REPETITIONS répétitions ; seuil du test t corrigé
N_PLIS, N_REPETITIONS = 5, 10
SEUIL_T = 2.0

VERSION_METHODE = "v8"
RAPPEL_MINIMAL = 0.60

## 1. Données, variables de la base et réglages figés

In [2]:
import ast
import numpy as np
import pandas as pd
from pathlib import Path
from IPython.display import Markdown, display

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 250)

# Racine du projet : premier dossier parent qui contient 'data'
current_dir = Path.cwd()
root_dir = next(p for p in [current_dir] + list(current_dir.parents) if (p / "data").exists())
train = pd.read_csv(root_dir / "data" / "ML" / "train_dataset.csv")
y = train["dpnm"].to_numpy()

# Variables de chaque base possible, rangées par encodage (copie des cellules « variables du scénario » de ml_14 et ml_15)
socle = {"num": ["LIMIT_BAL", "AGE"] + [f"PAY_{n}" for n in range(1, 7)], "log": [f"PAY_AMT{n}" for n in range(1, 7)],
         "log_signe": [f"BILL_AMT{n}" for n in range(1, 7)], "nom": ["SEX", "EDUCATION", "MARRIAGE"]}
BASES = {
    "14": {"retirees": ["AGE", "SEX", "EDUCATION", "MARRIAGE"],
           "ajout": {"num": ["CUMUL_INCIDENT"], "log": [], "log_signe": [], "nom": ["PAY_habituel"]}},
    "15": {"retirees": [],
           "ajout": {"num": ["ratio_PAY_BILL_global", "ratio_PAY_BILL_median", "ratio_PAY_BILL_regularite", "ratio_PAY_BILL1", "ratio_PAY_BILL2",
                             "ratio_PAY_BILL3", "ratio_PAY_BILL4", "ratio_PAY_BILL5", "ratio_BILL_LIMIT1", "ratio_BILL_LIMIT2", "ratio_BILL_LIMIT3",
                             "ratio_BILL_LIMIT4", "ratio_BILL_LIMIT5", "ratio_BILL_LIMIT6", "NB_MOIS_CTX", "CUMUL_INCIDENT"],
                     "log": ["BILL_evol_relatif", "PAYAMT_evol_relatif"], "log_signe": ["BILL_evol_brut", "PAYAMT_evol_brut"],
                     "nom": ["FLAG_OUVERTURE", "FLAG_DEGEL", "MOIS_ACTIVATION", "TYPE_USAGE", "PAY_habituel", "FAUX_CODAGE", "SURVEILLANCE_RECENTE",
                             "FLAG_CTX", "MOIS_SORTIE_CTX", "FLAG_RETARD", "MOIS_SORTIE_RETARD"]}},
}
# Remplacements décidés dans docs/colonnes_creees.md (les mêmes que dans les scénarios)
remplacements = {
    "ratio_PAY_BILL_global": ([np.nan], 100),
    "ratio_PAY_BILL_median": ([-1], 100),
    "ratio_PAY_BILL_regularite": ([np.nan, -1], 100),
}

def types_de(base, avec_groupe):
    # Colonnes par encodage pour la base, avec ou sans le groupe (le groupe : catégories 0 / 1, une colonne par catégorie)
    b = BASES[base]
    types = {t: [c for c in socle[t] if c not in b["retirees"]] + b["ajout"][t] for t in socle}
    types = {t: [c for c in cols if c not in GROUPE] for t, cols in types.items()}
    if avec_groupe:
        types["nom"] = types["nom"] + GROUPE
    return types

groupe_dans_base = all(c in sum(BASES[BASE]["ajout"].values(), []) for c in GROUPE)
VERSIONS = {"sans le groupe": types_de(BASE, False), "avec le groupe": types_de(BASE, True)}
for nom, types in VERSIONS.items():
    colonnes = sum(types.values(), [])
    absentes = [c for c in colonnes if c not in train.columns]
    assert not absentes, f"Colonnes absentes du train : {absentes}"
part = (train[GROUPE] != 0).any(axis=1).mean()
display(Markdown(f"**Base** : `ml_{BASE}` ; **test** : {'retrait' if groupe_dans_base else 'ajout'} du groupe {', '.join(f'`{c}`' for c in GROUPE)} ; "
                 f"clients concernés (une des colonnes différente de 0) : **{part * 100:.2f} %** du train ; "
                 f"variables : {len(sum(VERSIONS['sans le groupe'].values(), []))} sans le groupe, {len(sum(VERSIONS['avec le groupe'].values(), []))} avec."))

# Réglages figés : ceux retenus par la boucle dans la base, relus dans le fichier de résultats
resultats = pd.read_csv(root_dir / "data" / "ML" / "resultats_scenarios.csv")
lignes_base = resultats[(resultats["scenario"] == f"ml_{BASE}") & (resultats["version_methode"] == VERSION_METHODE)]
assert not lignes_base.empty, f"ml_{BASE} n'a pas de résultats enregistrés avec la méthode {VERSION_METHODE} : le lancer d'abord"
assert (lignes_base["rappel_minimal"] == RAPPEL_MINIMAL).all()

def lire_reglage(texte):
    # « depth 4 ; iterations 297 ; … » -> {"depth": 4, "iterations": 297, …} (texte gardé tel quel s'il n'est pas un nombre ou un tuple)
    reglage = {}
    for morceau in texte.split(" ; "):
        nom, valeur = morceau.split(" ", 1)
        try:
            reglage[nom] = ast.literal_eval(valeur)
        except (ValueError, SyntaxError):
            reglage[nom] = valeur
    return reglage

REGLAGES = {ligne["modele"]: lire_reglage(ligne["reglage"]) for _, ligne in lignes_base.iterrows()}
display(pd.DataFrame({"Modèle": list(REGLAGES), "Réglage figé (relu dans le fichier de résultats)": [lignes_base.set_index("modele").loc[m, "reglage"] for m in REGLAGES]}).set_index("Modèle"))

**Base** : `ml_14` ; **test** : ajout du groupe `FAUX_CODAGE`, `SURVEILLANCE_RECENTE` ; clients concernés (une des colonnes différente de 0) : **0.73 %** du train ; variables : 21 sans le groupe, 23 avec.

,Réglage figé (relu dans le fichier de résultats)
Modèle,
CatBoost,depth 4 ; iterations 297 ; l2_leaf_reg 6.67 ; ...
RandomForest,max_depth 6 ; max_features sqrt ; min_samples_...
LogisticRegression,C 0.1 ; l1_ratio 0.0
MLPClassifier,"alpha 1.0 ; hidden_layer_sizes (128, 64)"


In [3]:
from catboost import CatBoostClassifier
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler

scale_pos = (y == 0).sum() / (y == 1).sum()
TRANSFORMATIONS = {
    "num": lambda: StandardScaler(),
    "log": lambda: make_pipeline(FunctionTransformer(np.log1p, feature_names_out="one-to-one"), StandardScaler()),
    "log_signe": lambda: make_pipeline(FunctionTransformer(np.arcsinh, feature_names_out="one-to-one"), StandardScaler()),
    "nom": lambda: OneHotEncoder(sparse_output=False, handle_unknown="ignore"),
}

def avec_remplacement(colonne, transformation):
    valeurs, valeur = remplacements[colonne]
    valeurs = sorted(valeurs, key=lambda v: not pd.isna(v))
    return make_pipeline(*[SimpleImputer(missing_values=v, strategy="constant", fill_value=valeur) for v in valeurs], transformation)

def preprocesseur(types):
    # Le pipeline des scénarios (méthode v8), limité aux colonnes de la version
    return ColumnTransformer(transformers=
        [(t, TRANSFORMATIONS[t](), [c for c in cols if c not in remplacements]) for t, cols in types.items() if [c for c in cols if c not in remplacements]]
        + [(f"remplacement_{c}", avec_remplacement(c, TRANSFORMATIONS[t]()), [c]) for t, cols in types.items() for c in cols if c in remplacements])

def modele(nom, types):
    # Modèle aux réglages figés de la base ; un seul thread par modèle, les plis tournant en parallèle
    r = REGLAGES[nom]
    estimateurs = {
        "LogisticRegression": lambda: LogisticRegression(solver="saga", max_iter=5000, class_weight="balanced", random_state=42, **r),
        "MLPClassifier": lambda: MLPClassifier(max_iter=500, random_state=42, early_stopping=True, **r),
        "RandomForest": lambda: RandomForestClassifier(class_weight="balanced", random_state=42, n_jobs=1, **r),
        "CatBoost": lambda: CatBoostClassifier(scale_pos_weight=scale_pos, random_state=42, verbose=0, thread_count=1, allow_writing_files=False, **r),
    }
    return make_pipeline(preprocesseur(types), estimateurs[nom]())

## 2. Validation croisée répétée : les deux versions, sur les mêmes plis

Pour chaque modèle et chaque version (avec et sans le groupe), le modèle est entraîné sur chacun des 50 découpages (5 plis × 10 répétitions) et jugé sur le pli de validation : score décisionnel F2 au rappel minimal de 60 %, et précision moyenne. Les deux versions voient exactement les mêmes clients dans chaque pli : leurs scores se comparent pli par pli. Une ligne s'affiche par modèle, avec sa durée.

In [4]:
import time
from joblib import Parallel, delayed
from sklearn.metrics import average_precision_score, fbeta_score
from sklearn.model_selection import RepeatedStratifiedKFold

def seuil_pour_rappel(y_vrai, proba, rappel=RAPPEL_MINIMAL):
    proba_defauts = np.sort(proba[y_vrai == 1])[::-1]
    return proba_defauts[int(np.ceil(rappel * len(proba_defauts))) - 1]

def f2_au_rappel_minimal(y_vrai, proba):
    return fbeta_score(y_vrai, proba >= seuil_pour_rappel(y_vrai, proba), beta=2)

plis = list(RepeatedStratifiedKFold(n_splits=N_PLIS, n_repeats=N_REPETITIONS, random_state=42).split(train, y))

def un_pli(nom, types, entrainement, validation):
    colonnes = sum(types.values(), [])
    m = modele(nom, types).fit(train.iloc[entrainement][colonnes], y[entrainement])
    p = m.predict_proba(train.iloc[validation][colonnes])[:, 1]
    return f2_au_rappel_minimal(y[validation], p), average_precision_score(y[validation], p)

scores = {}   # (version, modèle) -> {"F2": [...], "Précision moyenne": [...]}
for nom in REGLAGES:
    debut = time.time()
    for version, types in VERSIONS.items():
        res = Parallel(n_jobs=-1)(delayed(un_pli)(nom, types, e, v) for e, v in plis)
        scores[(version, nom)] = {"F2": np.array([r[0] for r in res]), "Précision moyenne": np.array([r[1] for r in res])}
    print(f"{nom} : F2 sans le groupe {scores[('sans le groupe', nom)]['F2'].mean():.4f}, avec {scores[('avec le groupe', nom)]['F2'].mean():.4f} "
          f"({(time.time() - debut) / 60:.1f} min)")

CatBoost : F2 sans le groupe 0.4844, avec 0.4848 (0.3 min)
RandomForest : F2 sans le groupe 0.4833, avec 0.4833 (0.4 min)
LogisticRegression : F2 sans le groupe 0.4817, avec 0.4828 (0.1 min)
MLPClassifier : F2 sans le groupe 0.4800, avec 0.4804 (0.2 min)


## 3. Verdict : test t corrigé de Nadeau et Bengio

Pour chaque modèle, puis en meilleur contre meilleur : le gain moyen (avec le groupe moins sans), son écart-type sur les 50 plis, le nombre de plis gagnés, et le **t corrigé** :

t = gain moyen / racine( (1 / 50 + 1 / 4) × variance des gains )

Le terme 1 / 4 (taille d'un pli de validation rapportée à celle de la partie d'entraînement) corrige la corrélation entre plis qui réutilisent les mêmes clients. **Gain réel** si t ≥ 2, **perte** si t ≤ -2, sinon **dans le bruit**. Le meilleur contre meilleur décide (règle fixée en tête) ; les lignes par modèle et la précision moyenne sont données pour information.

In [5]:
CORRECTION = 1 / len(plis) + 1 / (N_PLIS - 1)   # Nadeau et Bengio : 1/J + taille de validation / taille d'entraînement

def t_corrige(gains):
    variance = gains.var(ddof=1)
    return gains.mean() / np.sqrt(CORRECTION * variance) if variance > 0 else 0.0

def verdict(t):
    return "✅ gain réel" if t >= SEUIL_T else ("❌ perte" if t <= -SEUIL_T else "➖ dans le bruit")

def meilleur(version):
    return max(REGLAGES, key=lambda nom: scores[(version, nom)]["F2"].mean())

lignes = []
for libelle, (sans, avec) in [(f"Meilleur contre meilleur ({meilleur('sans le groupe')} → {meilleur('avec le groupe')})",
                               (meilleur("sans le groupe"), meilleur("avec le groupe")))] + [(nom, (nom, nom)) for nom in REGLAGES]:
    ligne = {"Comparaison": libelle}
    for mesure in ("F2", "Précision moyenne"):
        gains = scores[("avec le groupe", avec)][mesure] - scores[("sans le groupe", sans)][mesure]
        t = t_corrige(gains)
        ligne[f"{mesure} sans"] = round(scores[("sans le groupe", sans)][mesure].mean(), 4)
        ligne[f"{mesure} avec"] = round(scores[("avec le groupe", avec)][mesure].mean(), 4)
        ligne[f"{mesure} : gain"] = f"{gains.mean():+.4f} ± {gains.std(ddof=1):.4f}"
        ligne[f"{mesure} : plis gagnés"] = f"{(gains > 0).sum()} sur {len(gains)}"
        ligne[f"{mesure} : t corrigé"] = round(t, 2)
        ligne[f"{mesure} : verdict"] = verdict(t)
    lignes.append(ligne)
tableau = pd.DataFrame(lignes).set_index("Comparaison")
display(Markdown(f"**Avec le groupe contre sans, pli par pli ({len(plis)} plis, base `ml_{BASE}`, réglages figés)**"))
display(tableau[[c for c in tableau.columns if c.startswith("F2")]])
display(tableau[[c for c in tableau.columns if c.startswith("Précision moyenne")]])
decision = tableau.iloc[0]["F2 : verdict"]
display(Markdown(f"**Décision (meilleur contre meilleur, score décisionnel F2)** : {decision} → "
                 + ("**groupe gardé**" if decision == "✅ gain réel" else "**groupe écarté définitivement**")
                 + (" (il reste dans la base)" if groupe_dans_base and decision == "✅ gain réel" else " (à retirer de la base)" if groupe_dans_base else "")))

**Avec le groupe contre sans, pli par pli (50 plis, base `ml_14`, réglages figés)**

,F2 sans,F2 avec,F2 : gain,F2 : plis gagnés,F2 : t corrigé,F2 : verdict
Comparaison,,,,,,
Meilleur contre meilleur (CatBoost → CatBoost),0.4844,0.4848,+0.0003 ± 0.0020,23 sur 50,0.33,➖ dans le bruit
CatBoost,0.4844,0.4848,+0.0003 ± 0.0020,23 sur 50,0.33,➖ dans le bruit
RandomForest,0.4833,0.4833,+0.0000 ± 0.0021,28 sur 50,0.04,➖ dans le bruit
LogisticRegression,0.4817,0.4828,+0.0012 ± 0.0018,42 sur 50,1.24,➖ dans le bruit
MLPClassifier,0.4800,0.4804,+0.0004 ± 0.0066,23 sur 50,0.12,➖ dans le bruit


,Précision moyenne sans,Précision moyenne avec,Précision moyenne : gain,Précision moyenne : plis gagnés,Précision moyenne : t corrigé,Précision moyenne : verdict
Comparaison,,,,,,
Meilleur contre meilleur (CatBoost → CatBoost),0.3359,0.3365,+0.0006 ± 0.0018,35 sur 50,0.68,➖ dans le bruit
CatBoost,0.3359,0.3365,+0.0006 ± 0.0018,35 sur 50,0.68,➖ dans le bruit
RandomForest,0.3337,0.3339,+0.0002 ± 0.0013,28 sur 50,0.28,➖ dans le bruit
LogisticRegression,0.3327,0.3340,+0.0013 ± 0.0017,40 sur 50,1.50,➖ dans le bruit
MLPClassifier,0.3304,0.3314,+0.0010 ± 0.0067,26 sur 50,0.29,➖ dans le bruit


**Décision (meilleur contre meilleur, score décisionnel F2)** : ➖ dans le bruit → **groupe écarté définitivement**

**Comment lire** : un gain positif veut dire que le groupe améliore le score. Avec 50 plis, l'écart-type de la moyenne des gains est bien plus petit que dans les scénarios ; le t corrigé dit si le gain moyen dépasse ce que le hasard des découpages peut produire. Le groupe concerne très peu de clients : même un vrai effet reste petit sur le score de l'ensemble du train. La décision s'écrit ensuite dans le journal `tableau_ML.md`, sur la ligne de `ml_2`.